In [ ]:
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from  sklearn.model_selection import train_test_split
import re

In [ ]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
sw=stopwords.words("english")

In [ ]:
pos= pd.read_csv("/content/pos.txt",sep="/t",header=None,encoding="latin-1",on_bad_lines="skip")


/tmp/ipykernel_2344/1823329588.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  pos= pd.read_csv("/content/pos.txt",sep="/t",header=None,encoding="latin-1",on_bad_lines="skip")


In [ ]:
pos=pd.concat([pos,pd.Series(np.ones(pos.shape[0]))],ignore_index=True,axis=1)
pos.columns=["reviews","mood"]


In [ ]:
pos

,reviews,mood
0,the rock is destined to be the 21st century's ...,1.0
1,"the gorgeously elaborate continuation of "" the...",1.0
2,effective but too-tepid biopic,1.0
3,if you sometimes like to go to the movies to h...,1.0
4,"emerges as something rare , an issue movie tha...",1.0
...,...,...
5323,both exuberantly romantic and serenely melanch...,1.0
5324,mazel tov to a film about a family's joyous li...,1.0
5325,standing in the shadows of motown is the best ...,1.0
5326,it's nice to see piscopo again after all these...,1.0


In [ ]:
neg=pd.read_csv("/content/negative.txt",sep="/t",header=None,encoding="latin-1",on_bad_lines="skip")

/tmp/ipykernel_2344/3176446575.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  neg=pd.read_csv("/content/negative.txt",sep="/t",header=None,encoding="latin-1",on_bad_lines="skip")


In [ ]:
neg=pd.concat([neg,pd.Series(np.zeros(neg.shape[0]))],ignore_index=True,axis=1)

In [ ]:
neg.columns=["reviews","mood"]

In [ ]:
neg

,reviews,mood
0,"simplistic , silly and tedious.",0.0
1,"it's so laddish and juvenile , only teenage bo...",0.0
2,exploitative and largely devoid of the depth o...,0.0
3,[garbus] discards the potential for pathologic...,0.0
4,a visually flashy but narratively opaque and e...,0.0
...,...,...
5324,a terrible movie that some people will neverth...,0.0
5325,there are many definitions of 'time waster' bu...,0.0
5326,"as it stands , crocodile hunter has the hurrie...",0.0
5327,the thing looks like a made-for-home-video qui...,0.0


In [ ]:
combined_review=pd.concat([pos,neg],axis=0).reset_index()

In [ ]:
combined_review

,index,reviews,mood
0,0,the rock is destined to be the 21st century's ...,1.0
1,1,"the gorgeously elaborate continuation of "" the...",1.0
2,2,effective but too-tepid biopic,1.0
3,3,if you sometimes like to go to the movies to h...,1.0
4,4,"emerges as something rare , an issue movie tha...",1.0
...,...,...,...
10652,5324,a terrible movie that some people will neverth...,0.0
10653,5325,there are many definitions of 'time waster' bu...,0.0
10654,5326,"as it stands , crocodile hunter has the hurrie...",0.0
10655,5327,the thing looks like a made-for-home-video qui...,0.0


In [ ]:
combined_review.loc[:,"reviews"] = combined_review.loc[:,"reviews"].apply(lambda x:x.lower())
combined_review.loc[:,'reviews'] = combined_review.loc[:,'reviews'].apply(lambda x:re.sub(r"[^\w\s]"," ",x))
combined_review.loc[:,'reviews'] = combined_review.loc[:,'reviews'].apply(lambda x:' '.join([word for word in x.split() if word not in (sw)]))

In [ ]:
combined_review

,index,reviews,mood
0,0,rock destined 21st century new conan going mak...,1.0
1,1,gorgeously elaborate continuation lord rings t...,1.0
2,2,effective tepid biopic,1.0
3,3,sometimes like go movies fun wasabi good place...,1.0
4,4,emerges something rare issue movie honest keen...,1.0
...,...,...,...
10652,5324,terrible movie people nevertheless find moving,0.0
10653,5325,many definitions time waster movie must surely...,0.0
10654,5326,stands crocodile hunter hurried badly cobbled ...,0.0
10655,5327,thing looks like made home video quickie,0.0


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(combined_review['reviews'].values,combined_review['mood'].values, test_size = 0.33, random_state = 42)

In [ ]:
train_data=pd.DataFrame({"reviews":X_train,"mood":Y_train})
test_data=pd.DataFrame({"reviews":X_test,"mood":Y_test})

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer()

In [ ]:
train_vector=vectorizer.fit_transform(train_data["reviews"])
test_vector=vectorizer.transform(test_data["reviews"])

In [ ]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression()
model.fit(train_vector,train_data["mood"])

LogisticRegression()

In [ ]:
logistic_prediction=model.predict(test_vector)

In [ ]:
from sklearn.metrics import classification_report
report=classification_report(test_data["mood"],logistic_prediction,output_dict=True)

In [ ]:
print("positive:",report['1.0']['recall'])
print("Negative:",report['0.0']['recall'])

positive: 0.744225352112676
Negative: 0.7600459242250287


In [ ]:
from sklearn.metrics import accuracy_score
print("accuracy:",accuracy_score(test_data["mood"],logistic_prediction))

accuracy: 0.752061415979528


In [ ]:
print(classification_report(test_data["mood"],logistic_prediction))

              precision    recall  f1-score   support

         0.0       0.74      0.76      0.75      1742
         1.0       0.76      0.74      0.75      1775

    accuracy                           0.75      3517
   macro avg       0.75      0.75      0.75      3517
weighted avg       0.75      0.75      0.75      3517



In [ ]:
from sklearn import svm
linear_svm=svm.SVC(kernel="linear")
linear_svm.fit(train_vector,train_data["mood"])

SVC(kernel='linear')

In [ ]:
svm_prediction=linear_svm.predict(test_vector)

In [ ]:
print(classification_report(test_data["mood"],logistic_prediction))

              precision    recall  f1-score   support

         0.0       0.74      0.76      0.75      1742
         1.0       0.76      0.74      0.75      1775

    accuracy                           0.75      3517
   macro avg       0.75      0.75      0.75      3517
weighted avg       0.75      0.75      0.75      3517

